# Занятие 2, пара 2. Калибровка камеры

Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai

В прошлой паре посмотрели, как появляется изображение. Теперь разберёмся, почему камера иногда изгибает прямые линии и что с этим можно сделать. Большой кусок кода уже готов — писать калибровку с нуля сегодня не будем. Зато запустим её, поменяем несколько вещей и проверим, что получилось. В конце сохраним параметры камеры, картинки до/после и свой вывод. Эта тетрадь запускается отдельно от предыдущей.

In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check, summary

PROJECT_DIR


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from course_support.cvtools import imread, imwrite, show
CAMERA_DIR = DATA_DIR / "camera"
OUTPUT_DIR = PROJECT_DIR / "reports" / "camera"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
paths = sorted(CAMERA_DIR.glob("board_*.png"))
image = imread(paths[0])
show(image, titles=["Прямую доску искривил учебный объектив"])

## Дисторсия и перспектива

При перспективе прямые линии остаются прямыми, но параллельные могут сходиться. Дисторсия объектива может изгибать прямые. `undistort` исправляет модель искажений, а не поворачивает доску лицом к нам. Чёрные области по краям после исправления возможны: там не было исходных пикселей.

## Ищем внутренние углы

На доске 10×7 клеток, но внутренних пересечений 9×6. Алгоритму задаём именно **внутренние углы**. Давайте попробуем на одном кадре: found скажет True или False — получилось ли найти углы. А в corners будут их координаты в пикселях.

In [ ]:
pattern = (9, 6)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
found, corners = cv2.findChessboardCorners(gray, pattern)
marked = image.copy()
if found:
    cv2.drawChessboardCorners(marked, pattern, corners, found)
print("Найдены:", found, "Количество:", len(corners) if found else 0)
show(marked, titles=["54 внутренних угла"])

### ✏️ Ваш ход 1 — другой кадр

Откройте board_03.png, переведите в серый, найдите углы при pattern=(9,6). Сохраните `found_other` и `corners_other`, покажите размеченный кадр.

In [ ]:
# ✏️ ваш код здесь

In [ ]:
check('02.2.1', found_other, corners_other)

## Калибруем — готовый рецепт

Нужны несколько ракурсов и соответствия «координаты угла на плоской доске ↔ пиксель на снимке». Сторона клетки 0.025 м. Числа K описывают фокус и центр, dist — искажения. Функция подбирает их по измеренным углам. Для этой учебной модели фиксируем k3=0 (CALIB_FIX_K3), чтобы не подгонять лишний коэффициент под шум детектора; это не универсальная настройка для всех объективов.

Это **синтетические**, специально подготовленные кадры. Маленькая ошибка здесь не обещает такую же точность на реальном телефоне.

In [ ]:
object_grid = np.zeros((9*6, 3), np.float32)
object_grid[:, :2] = np.mgrid[0:9, 0:6].T.reshape(-1, 2) * 0.025
object_points, image_points, used_names = [], [], []
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
for path in paths:
    frame = imread(path)
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    ok, pts = cv2.findChessboardCorners(gray_frame, pattern)
    if ok:
        refined = cv2.cornerSubPix(gray_frame, pts, (5,5), (-1,-1), criteria)
        object_points.append(object_grid.copy())
        image_points.append(refined)
        used_names.append(path.name)
print("Использовано кадров:", len(used_names), "из", len(paths))
# Резерв: измерения детектора с тех же кадров, без повторного поиска.
if len(used_names) < 6:
    cached = np.load(CAMERA_DIR / "detected_corners.npz")
    object_points = list(cached["object_points"])
    image_points = list(cached["image_points"])
    used_names = list(cached["names"])
    print("Используем предоставленные измерения; отметьте это в отчёте")
image_size = (image.shape[1], image.shape[0])
rms, K, dist, rvecs, tvecs = cv2.calibrateCamera(object_points, image_points, image_size, None, None, flags=cv2.CALIB_FIX_K3)
print("RMS в пикселях:", rms)
print("K:\n", K)
print("dist:", dist.ravel())

## До и после

In [ ]:
grid_image = imread(CAMERA_DIR / "grid.png")
corrected = cv2.undistort(grid_image, K, dist)
show(grid_image, corrected, titles=["До", "После"])
imwrite(OUTPUT_DIR / "before.png", grid_image)
imwrite(OUTPUT_DIR / "after.png", corrected)
np.savez(OUTPUT_DIR / "calibration.npz", K=K, dist=dist)

### ✏️ Ваш ход 2 — применить параметры

Исправьте board_03.png с помощью K и dist. Сохраните результат в `fixed_other`, покажите до/после. Запишите, что изменилось и почему доска всё ещё наклонена.

In [ ]:
# ✏️ ваш код здесь

In [ ]:
check('02.2.2', source_other, fixed_other, K, dist)

## Ошибка: куда модель поставила углы

Спроецируем углы назад в изображение и сравним с измеренными точками. Используем RMS: корень из среднего квадрата расстояния в пикселях. Считаем одинаковым способом для каждого кадра. Это ошибка на кадрах калибровки, не независимая проверка качества.

In [ ]:
def reprojection_rms(observed, predicted):
    delta = observed.reshape(-1,2) - predicted.reshape(-1,2)
    return float(np.sqrt(np.mean(np.sum(delta**2, axis=1))))

errors = []
for obj, pts, rv, tv in zip(object_points, image_points, rvecs, tvecs):
    predicted, _ = cv2.projectPoints(obj, rv, tv, K, dist)
    errors.append(reprojection_rms(pts, predicted))
worst = int(np.argmax(errors))
print("Наибольшая ошибка:", used_names[worst], errors[worst], "px")
plt.bar(range(len(errors)), errors)
plt.xlabel("Кадр"); plt.ylabel("RMS, px"); plt.show()
show(imread(CAMERA_DIR / used_names[worst]), titles=["Кадр с максимальной ошибкой"])
predicted, _ = cv2.projectPoints(object_points[0], rvecs[0], tvecs[0], K, dist)
base_error = reprojection_rms(image_points[0], predicted)

### ✏️ Ваш ход 3 — испортить измерение

Создайте копию image_points[0], сдвиньте x первого угла на 20 пикселей и посчитайте `bad_error` относительно прежнего predicted. Сравните с base_error. Параметры камеры не переобучаем: проверяем влияние одной ошибки измерения.

In [ ]:
# ✏️ ваш код здесь

In [ ]:
check('02.2.3', base_error, bad_error)

## Практический запас: когда углы перестают находиться — 12–15 минут

Возьмите board_03.png и сравните исходник, сильное размытие и кадр с закрытой центральной частью доски. Для каждого запустите findChessboardCorners с (9,6). Покажите изображения и запишите найдено/не найдено.

Если на размытой картинке всё ещё найдено — это полезный результат, а не ошибка задания. Меняйте силу размытия и укажите реально полученные результаты. Необязательный опыт; новые библиотеки не нужны.

In [ ]:
# ✏️ ваш код здесь

### Ещё 5–8 минут: достаточно ли вызвать функцию?

Примените cv2.undistort к grid_image с матрицей K и нулевыми коэффициентами dist (`np.zeros_like(dist)`). Сравните с коррекцией найденными коэффициентами. Какой вариант почти повторяет исходник? Достаточно ли одного вызова функции, чтобы считать задачу решённой? Подпишите две картинки.

## ЛР1 и итог

Сохраните обе тетради и reports/camera. Условия ЛР1 — `docs/ЛР1.md`: предоставленных данных достаточно для полного балла. Свою камеру калибровать необязательно. Срок преподаватель объявляет при выдаче. Обычный ZIP и ссылка на GitHub равноправны.

Минимум сегодня: найти углы, получить картинку до/после, объяснить рост ошибки. Если не успели — допишите дома или в доступной аудитории. Баллы за участие отдельно не ставятся.

In [ ]:
summary('02.2')

## Если хочется глубже

Попробуйте повторную калибровку на шести разных кадрах и сравните K с полным набором. Меньшая ошибка на меньшем наборе не означает лучшую модель. Число ракурсов и покрытие кадра имеют значение. Необязательное расширение.

Документация: https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html

## На потом, без обязательного задания

[Урок OpenCV о калибровке](https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html): сопоставьте один наш вызов с документацией. Можно ограничиться пятью минутами. В книге [Шелиски](https://szeliski.org/Book/) ищите объяснение модели камеры, когда захотите понять математику глубже.